# LoRA Fine-Tune GPT-2 on Romanized Telugu
**TDL Project — Week 3**

This notebook:
1. Mounts Google Drive and reads `cleaned_data.txt`
2. Clones the project repo from GitHub
3. Installs dependencies
4. Tokenizes the dataset (GPT-2 tokenizer, max_length=128)
5. Fine-tunes GPT-2 with LoRA (r=8) for 5 epochs on a T4 GPU
6. Evaluates on 15 Week-1 Telugu sentences (before/after PPL)
7. Saves model + results back to Drive

**Before running:**
- Upload `cleaned_data.txt` to `My Drive/TDL-Project/data/cleaned_data.txt`
- Runtime → Change runtime type → **T4 GPU**
- Then: Runtime → Run all

In [ ]:
# ── 0. Verify GPU ─────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU found — change runtime to T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT   = '/content/drive/MyDrive/TDL-Project'
CLEANED_DATA = f'{DRIVE_ROOT}/data/cleaned_data.txt'

assert os.path.exists(CLEANED_DATA), (
    f"cleaned_data.txt not found at {CLEANED_DATA}\n"
    f"Upload it to: My Drive/TDL-Project/data/cleaned_data.txt"
)
lines = open(CLEANED_DATA).readlines()
print(f"cleaned_data.txt found — {len(lines):,} lines")

In [ ]:
# ── 2. Clone repo & install deps ──────────────────────────────────────────
REPO_URL = 'https://github.com/Deekshithrathod/TDL-Project.git'
REPO_DIR = '/content/TDL-Project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print('Working dir:', os.getcwd())

In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────────
!pip install -q transformers datasets accelerate peft tokenizers emoji nltk tabulate

# Download NLTK words corpus (used by train_tokenizer.py if needed)
import nltk
nltk.download('words', quiet=True)
print('Dependencies installed.')

In [ ]:
# ── 4. Copy cleaned_data.txt into repo data dir ───────────────────────────
import shutil
os.makedirs('data/processed', exist_ok=True)
shutil.copy(CLEANED_DATA, 'data/processed/cleaned_data.txt')
print('Copied cleaned_data.txt → data/processed/cleaned_data.txt')

In [ ]:
# ── 5. Tokenize dataset (GPT-2, max_length=128) ───────────────────────────
!python scripts/prepare_dataset.py \
    --input   data/processed/cleaned_data.txt \
    --output  data/clm_dataset_gpt2 \
    --tokenizer gpt2 \
    --max_length 128

In [ ]:
# ── 6. Fine-tune GPT-2 with LoRA ──────────────────────────────────────────
# T4 has 16 GB VRAM — batch_size=16 fits comfortably.
# fp16=True is safe on CUDA T4 and speeds up training ~2x.
!python scripts/finetune_gpt2_lora.py \
    --dataset    data/clm_dataset_gpt2 \
    --output     models/gpt2_lora_finetuned \
    --epochs     5 \
    --rank       8 \
    --batch_size 16

In [ ]:
# ── 7. Show results ───────────────────────────────────────────────────────
import pandas as pd

print('=== Perplexity Curve ===')
curve = pd.read_csv('report/perplexity_curve.csv')
print(curve.to_string(index=False))

print('\n=== Before / After Comparison (Telugu sentences) ===')
comp = pd.read_csv('report/finetune_comparison.csv')
print(comp[['sentence','pretrained_ppl','finetuned_ppl','ppl_delta']].to_string(index=False))

improved = (comp['finetuned_ppl'] < comp['pretrained_ppl']).sum()
print(f'\nSentences improved: {improved}/{len(comp)}')
print(f'Avg pretrained PPL: {comp["pretrained_ppl"].mean():.1f}')
print(f'Avg finetuned  PPL: {comp["finetuned_ppl"].mean():.1f}')

In [ ]:
# ── 8. Save everything back to Drive ──────────────────────────────────────
import shutil

OUT_DIR = f'{DRIVE_ROOT}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Model adapter weights
model_dst = f'{OUT_DIR}/gpt2_lora_finetuned'
if os.path.exists(model_dst):
    shutil.rmtree(model_dst)
shutil.copytree('models/gpt2_lora_finetuned', model_dst)
print(f'Model saved to Drive: {model_dst}')

# Report CSVs
for fname in ['perplexity_curve.csv', 'finetune_comparison.csv']:
    src = f'report/{fname}'
    dst = f'{OUT_DIR}/{fname}'
    shutil.copy(src, dst)
    print(f'Saved: {dst}')

# Checkpoints (trainer_state.json for the perplexity log)
ckpt_src = 'models/gpt2_lora_finetuned_checkpoints'
if os.path.exists(ckpt_src):
    state_src = None
    for root, dirs, files in os.walk(ckpt_src):
        if 'trainer_state.json' in files:
            state_src = os.path.join(root, 'trainer_state.json')
            break
    if state_src:
        shutil.copy(state_src, f'{OUT_DIR}/trainer_state.json')
        print(f'Saved: {OUT_DIR}/trainer_state.json')

print('\nAll outputs saved to Drive.')

## After training completes

Download from Drive:
- `TDL-Project/outputs/gpt2_lora_finetuned/` → copy to `models/gpt2_lora_finetuned/` locally
- `TDL-Project/outputs/perplexity_curve.csv` → copy to `report/`
- `TDL-Project/outputs/finetune_comparison.csv` → copy to `report/`

Then commit:
```bash
git add models/gpt2_lora_finetuned/ report/perplexity_curve.csv report/finetune_comparison.csv History.md
git commit -m "Add LoRA fine-tuning results (5 epochs, T4 GPU)"
git push
```